In [2]:
import pandas as pd
import numpy as np

# ============================================================
# Configuration
# ============================================================
NPI_FILE       = 'radiologist_pathologist_filtered.csv'
CROSSWALK_FILE = 'ZIP_COUNTY_122025.xlsx'
OUTPUT_FILE    = 'county_radiologist_pathologist_counts.csv'
EXCLUDED_FILE  = 'county_counts_excluded_ambiguous.csv'

PRIMARY_CODE      = '2085R0202X'
PATHOLOGIST_CODES = {'207ZP0102X', '207ZP0101X', '207ZC0500X'}
ALL_TARGET_CODES  = {PRIMARY_CODE} | PATHOLOGIST_CODES

US_STATES = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','DC','FL','GA','HI','ID',
    'IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO',
    'MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA',
    'RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY'
}

TAXONOMY_COLS       = [f'Healthcare Provider Taxonomy Code_{i}' for i in range(1, 16)]
TAXONOMY_SWITCH_COLS = [f'Healthcare Provider Primary Taxonomy Switch_{i}' for i in range(1, 16)]

# ============================================================
# Step 1: Load data
# ============================================================
print("Loading data...")
npi   = pd.read_csv(NPI_FILE, low_memory=False, dtype=str)
xwalk = pd.read_excel(CROSSWALK_FILE, dtype=str)

for col in ['RES_RATIO', 'BUS_RATIO', 'OTH_RATIO', 'TOT_RATIO']:
    xwalk[col] = pd.to_numeric(xwalk[col], errors='coerce').fillna(0)

print(f"NPI records loaded: {len(npi):,}")

# ============================================================
# Step 2: Handle ambiguous providers
# Rule: providers with BOTH radiologist AND pathologist codes
#   → if primary taxonomy is one of the target codes: assign by primary
#   → if primary taxonomy is NOT in target codes: exclude from all groups
#
# Providers with only ONE group's code are always kept
# regardless of whether primary=Y or N
# ============================================================

has_rad  = npi[TAXONOMY_COLS].isin({PRIMARY_CODE}).any(axis=1)
has_path = npi[TAXONOMY_COLS].isin(PATHOLOGIST_CODES).any(axis=1)
has_both = has_rad & has_path

print(f"\nProviders with radiologist code only: {(has_rad & ~has_path).sum():,}")
print(f"Providers with pathologist code only: {(has_path & ~has_rad).sum():,}")
print(f"Providers with BOTH codes:            {has_both.sum():,}")

# For providers with both codes, check if primary is in target codes
def get_primary_target(row):
    """Return the primary target code if it exists, else None"""
    for i, col in enumerate(TAXONOMY_COLS):
        if row[col] in ALL_TARGET_CODES:
            switch_col = TAXONOMY_SWITCH_COLS[i]
            if row[switch_col] == 'Y':
                return row[col]
    return None

ambiguous = npi[has_both].copy()
ambiguous['primary_target'] = ambiguous.apply(get_primary_target, axis=1)

# Split ambiguous into: resolvable (primary found) vs excluded (no primary)
resolvable = ambiguous[ambiguous['primary_target'].notna()].copy()
excluded   = ambiguous[ambiguous['primary_target'].isna()].copy()

print(f"\nOf the {has_both.sum()} providers with both codes:")
print(f"  Resolvable (primary found):     {len(resolvable):,}")
print(f"  Excluded (no primary in target): {len(excluded):,}")

# Save excluded providers for records
excluded['exclusion_reason'] = 'ambiguous_dual_specialty_no_primary'
excluded.to_csv(EXCLUDED_FILE, index=False)
print(f"\nExcluded providers saved to: {EXCLUDED_FILE}")

# ============================================================
# Step 3: Build clean NPI subsets
# ============================================================

# Start with non-ambiguous providers
npi_single = npi[~has_both].copy()

# Add resolvable ambiguous providers with corrected matched_taxonomy
resolvable['matched_taxonomy'] = resolvable['primary_target']
npi_clean = pd.concat([npi_single, resolvable], ignore_index=True)

print(f"\nClean NPI records for analysis: {len(npi_clean):,}")
print(f"(Original: {len(npi):,} | Excluded: {len(excluded):,})")

# Subset by group
npi_rad         = npi_clean[npi_clean['matched_taxonomy'] == PRIMARY_CODE].copy()
npi_rad_primary = npi_rad[npi_rad['matched_taxonomy_is_primary'] == 'Y'].copy()
npi_path        = npi_clean[npi_clean['matched_taxonomy'].isin(PATHOLOGIST_CODES)].copy()

print(f"\nDiagnostic Radiologists - all:          {len(npi_rad):,}")
print(f"Diagnostic Radiologists - primary only: {len(npi_rad_primary):,}")
print(f"Pathologists - all:                     {len(npi_path):,}")

# ============================================================
# Step 4: Normalize allocation ratios
# ============================================================
xwalk['BUS_RATIO_NORM'] = xwalk.groupby('ZIP')['BUS_RATIO'].transform(
    lambda x: x / x.sum() if x.sum() > 0 else x
)
xwalk['RES_RATIO_NORM'] = xwalk.groupby('ZIP')['RES_RATIO'].transform(
    lambda x: x / x.sum() if x.sum() > 0 else x
)
xwalk['ALLOC_RATIO'] = np.where(
    xwalk['BUS_RATIO_NORM'] > 0,
    xwalk['BUS_RATIO_NORM'],
    xwalk['RES_RATIO_NORM']
)

# ============================================================
# Step 5: Merge and aggregate function
# ============================================================
def merge_and_aggregate(npi_df, label):
    merged = npi_df.merge(
        xwalk[['ZIP', 'COUNTY', 'USPS_ZIP_PREF_STATE', 'ALLOC_RATIO']],
        left_on='zip5',
        right_on='ZIP',
        how='left'
    )

    unmatched = merged['COUNTY'].isna().sum()
    print(f"\n[{label}] Unmatched ZIPs: {unmatched:,} ({unmatched/len(npi_df)*100:.1f}%)")

    merged = merged.dropna(subset=['COUNTY'])
    merged['provider_count'] = merged['ALLOC_RATIO']

    county_counts = merged.groupby(
        ['COUNTY', 'USPS_ZIP_PREF_STATE']
    )['provider_count'].sum().reset_index()
    county_counts.columns = ['FIPS', 'state', f'count_{label}']
    county_counts[f'count_{label}'] = county_counts[f'count_{label}'].round(4)

    print(f"[{label}] Counties with >=1 provider: {len(county_counts):,}")
    print(f"[{label}] Total allocated count:       {county_counts[f'count_{label}'].sum():.1f}")

    return county_counts

# ============================================================
# Step 6: Run aggregation
# ============================================================
counts_rad         = merge_and_aggregate(npi_rad,         'radiologist')
counts_rad_primary = merge_and_aggregate(npi_rad_primary, 'radiologist_primary')
counts_path        = merge_and_aggregate(npi_path,        'pathologist')

# ============================================================
# Step 7: Combine into single county file
# ============================================================
county_df = counts_rad \
    .merge(counts_rad_primary, on=['FIPS', 'state'], how='outer') \
    .merge(counts_path,        on=['FIPS', 'state'], how='outer')

county_df = county_df.fillna(0)
county_df = county_df[county_df['state'].isin(US_STATES)]

print(f"\nFinal county count (50 states + DC): {len(county_df):,}")

# ============================================================
# Step 8: Summary
# ============================================================
print("\n=== Radiologist Count Distribution by County ===")
print(county_df['count_radiologist'].describe().round(2))

print(f"\nCounties with zero radiologists (main):         {(county_df['count_radiologist'] == 0).sum():,}")
print(f"Counties with zero radiologists (primary only): {(county_df['count_radiologist_primary'] == 0).sum():,}")
print(f"Counties with zero pathologists:                {(county_df['count_pathologist'] == 0).sum():,}")

# ============================================================
# Step 9: Save
# ============================================================
county_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved to: {OUTPUT_FILE}")
print(f"Columns: {county_df.columns.tolist()}")
print("\nNext step: merge with Census population data to compute density per 100k")

Loading data...
NPI records loaded: 69,066

Providers with radiologist code only: 48,188
Providers with pathologist code only: 20,872
Providers with BOTH codes:            6

Of the 6 providers with both codes:
  Resolvable (primary found):     4
  Excluded (no primary in target): 2

Excluded providers saved to: county_counts_excluded_ambiguous.csv

Clean NPI records for analysis: 69,064
(Original: 69,066 | Excluded: 2)

Diagnostic Radiologists - all:          48,190
Diagnostic Radiologists - primary only: 35,233
Pathologists - all:                     20,874

[radiologist] Unmatched ZIPs: 61 (0.1%)
[radiologist] Counties with >=1 provider: 1,895
[radiologist] Total allocated count:       47796.5

[radiologist_primary] Unmatched ZIPs: 46 (0.1%)
[radiologist_primary] Counties with >=1 provider: 1,805
[radiologist_primary] Total allocated count:       34953.4

[pathologist] Unmatched ZIPs: 33 (0.2%)
[pathologist] Counties with >=1 provider: 1,378
[pathologist] Total allocated count:     